<a href="https://colab.research.google.com/github/SiyumiJayawardhane/sinhala-wsd/blob/sinhala_textbook_extraction/textbook_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

1. Setup

*   Installs pymupdf (to read the PDF’s text) and pandukabhaya (to convert FM-Abhaya to Unicode), then connects Drive.

In [17]:
!pip install -q pymupdf pandukabhaya

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 41.0 MB/s eta 0:00:00


In [21]:
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, random, subprocess, unicodedata, datetime
from collections import Counter
from pandukabhaya import Converter

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


2. Load the book

In [10]:
# Why not pdftotext: (1) it reads two-column poems column by column, so a verse line loses its
# rhyme word; (2) it can't see fonts, so English words (Times font) were "converted" into fake Sinhala.
# PyMuPDF gives each text span with its font and x/y position, so we can fix both.
import pymupdf
COL_GAP, WORD_GAP, LINE_TOL = 12.0, 1.5, 3.0

def read_spans(path):
    """List of pages; each page = list of spans {text, font, bbox}."""
    with pymupdf.open(path) as doc:
        return [[s for b in p.get_text("dict")["blocks"] for l in b.get("lines", []) for s in l["spans"]
                 if s["text"].strip()] for p in doc]

def detect_encoding(pages):
    """'fm' if most text is in FM (legacy) fonts, 'unicode' if it is already Sinhala Unicode,
    'none' if there is (almost) no text layer -> scanned -> needs Document AI OCR."""
    chars = Counter()
    for p in pages:
        for s in p:
            chars["fm" if s["font"].startswith("FM") else
                  "unicode" if SINHALA.search(s["text"]) else "other"] += len(s["text"])
    total = sum(chars.values())
    if total < 500:
        return "none", chars
    return ("fm" if chars["fm"] > chars["unicode"] else "unicode"), chars

SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")
for b in BOOKS:
    b["spans"] = read_spans(b["pdf_path"])
    b["encoding"], chars = detect_encoding(b["spans"])
    fonts = Counter(s["font"] for p in b["spans"] for s in p)
    print(f'{b["book_id"]}: {len(b["spans"])} pages -> {b["encoding"]} | chars by type {dict(chars)}')
    print("   fonts:", fonts.most_common(8))

OK      /content/drive/MyDrive/Research-WSD-New/textbooks/sinhala lit g-10-11.pdf


3. Read the text and check its encoding

In [11]:
SINHALA = re.compile(r"[\u0D80-\u0DFF]")
LATIN   = re.compile(r"[A-Za-z]")

def pdf_pages(path):
    """pdftotext output split into pages (form feed = page break). Index 0 = PDF page 1."""
    raw = subprocess.run(["pdftotext", path, "-"], capture_output=True, text=True, check=True).stdout
    pages = raw.split("\f")
    return pages[:-1] if pages and not pages[-1].strip() else pages

def detect_encoding(pages):
    """'unicode' = already Sinhala Unicode; 'fm' = legacy font (Latin letters stand for Sinhala glyphs);
    'none' = no text layer (scanned) -> needs OCR (Document AI) instead."""
    text = "".join(pages)
    s, l = len(SINHALA.findall(text)), len(LATIN.findall(text))
    if s + l < 500:
        return "none", s, l
    return ("unicode" if s > l else "fm"), s, l

for b in BOOKS:
    b["raw_pages"] = pdf_pages(b["pdf_path"])
    b["encoding"], s, l = detect_encoding(b["raw_pages"])
    print(f'{b["book_id"]}: {len(b["raw_pages"])} pages | Sinhala chars {s:,} | Latin chars {l:,} -> {b["encoding"]}')
    print("   sample raw text:", re.sub(r"\s+", " ", b["raw_pages"][min(15, len(b["raw_pages"])-1)])[:150])

sinhala_lit_g10_11: 140 pages | Sinhala chars 0 | Latin chars 145,027 -> fm
   sample raw text: ye<,S .sh uyd id.rh fia fldam kue;s jd;fhka ye<,S .sh wgf<dia wlafI!ys‚hla fiakdfjka hqla; je kslauqKdy' flajÜgfhda o ;ukaf.a nia rÊcqrejka is;g kxjd 


4. Convert FM-Abhaya to Unicode

In [12]:
conv = Converter("fm_abhaya")
NOISE = re.compile(r"[\u00AD\u200B\uFEFF]")

# Glyphs the fm_abhaya table leaves unconverted in this book, found from cell 5's leftover list and
# checked on real words: `ඵ -> ළු (නෙළුම්, පුළුවන්, ඇතුළු), ¥ -> දූ (දූතයා, දූවිලි, ඕපාදූප),
# i after a Sinhala letter -> ද්‍ය (විද්‍යාලය, පද්‍ය); `ද -> ඳ (අඳින්නේ, පැළඳි); `ඩ -> ඬ (පඬිතුමා).
POST_FIXES = [(re.compile(r"`ඵ"), "ළු"), (re.compile(r"`ද"), "ඳ"), (re.compile(r"`ඩ"), "ඬ"),
              (re.compile(r"¥"), "දූ"),
              (re.compile(r"(?<=[\u0D80-\u0DFF])i"), "ද්\u200Dය")]

def to_unicode(page, encoding):
    if encoding == "fm":
        # In this book the FM glyph '“' is used as an opening quotation mark, but the fm_abhaya table
        # maps it to ර්ණ. Replace it BEFORE converting, so real ර්ණ words (වර්ණය, කර්ණ) are untouched.
        page = page.replace("“", "\u0001")
        page = conv.convert(page).replace("\u0001", "“")
        for pat, rep in POST_FIXES:
            page = pat.sub(rep, page)
    return NOISE.sub("", unicodedata.normalize("NFC", page))

for b in BOOKS:
    if b["encoding"] == "none":
        print(f'{b["book_id"]}: no text layer — skip here, use Document AI OCR'); continue
    b["quote_glyphs"] = sum(p.count("“") for p in b["raw_pages"]) if b["encoding"] == "fm" else 0
    b["pages"] = [to_unicode(p, b["encoding"]) for p in b["raw_pages"]]
    json.dump(b["pages"], open(f'{b["out"]}/{b["book_id"]}_pages_unicode.json', "w", encoding="utf-8"), ensure_ascii=False)
    print(f'{b["book_id"]}: {len(b["pages"])} pages converted | “ glyphs kept as quotes: {b["quote_glyphs"]}')
    print("   sample:", re.sub(r"\s+", " ", b["pages"][min(15, len(b["pages"])-1)])[:150])

sinhala_lit_g10_11: 140 pages converted | “ glyphs kept as quotes: 380
   sample: හැළලී ගිය මහා සාගරය සේ කෝප නමැති වාතයෙන් හැළලී ගිය අටළොස් අක්ෂෞහිණියක් සේනාවෙන් යුක්ත වැ නික්මුණාහ. කේවට්ටයෝ ද තමන්ගේ බස් රජ්ජුරුවන් සිතට නංවා ගත නොහී


5. Check the conversion page by page

In [13]:
# After a good conversion almost no Latin letters remain. Leftover Latin = glyphs the converter
# did not map (or real English text on that page) have to look at those pages before using them.
LATIN_LEFT_MAX = 0.02          # flag a page if > 2% of its letters are still Latin

rows = []
for b in BOOKS:
    if "pages" not in b: continue
    leftovers = Counter()
    for i, p in enumerate(b["pages"]):
        s, l = len(SINHALA.findall(p)), len(LATIN.findall(p))
        leftovers.update(ch for ch in p if not (SINHALA.match(ch) or ch.isspace() or ch.isdigit()
                                               or ch in ".,;:!?()[]-–—'\"“”‘’/\u200C\u200D"))
        rows.append({"book": b["book_id"], "pdf_page": i + 1, "sinhala": s, "latin": l,
                     "flag": (s + l) > 0 and l / (s + l) > LATIN_LEFT_MAX})
    flagged = [r["pdf_page"] for r in rows if r["book"] == b["book_id"] and r["flag"]]
    print(f'{b["book_id"]}: {len(flagged)} flagged pages {flagged[:20]}')
    print("   most common leftover characters:", leftovers.most_common(15))

import pandas as pd
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/conversion_page_check.csv", index=False, encoding="utf-8")

sinhala_lit_g10_11: 0 flagged pages []
   most common leftover characters: [('+', 27), ('=', 12), ('*', 8), ('•', 1), ('`', 1), ('⋆', 1)]


6. Split into sentences


*   Page-number lines are dropped.
*   Prose is split at . ? !.
*   Long poem chunks are cut into 2-line units, because poems have no full stops.

In [14]:
#  - lines that are only page numbers (digits / roman numerals) are dropped
#  - the book's study notes start with labels අන්වය : / අර්ථය : / විස්තර : -> each label starts a new unit,
#    so a verse stanza is never glued to its explanation
#  - prose is split after . ? !
#  - verse has no full stops: a chunk > 280 chars with >= 3 lines is cut into units of
#    VERSE_LINES lines
#  - units with fewer than 4 Sinhala letters are dropped (numbering "03.", initials "එම්.")
VERSE_LINES = 2
PAGE_NUM = re.compile(r"[\divxlcIVXLC\s]+")
SPLIT    = re.compile(r"(?<=[.?!])\s+")
LABEL    = re.compile(r"\s*(?=(?:අන්වය|අර්ථය|විස්තර)\s*\*?\s*:)")

def split_units(pages, offset):
    units = []
    for pi, p in enumerate(pages):
        lines = [re.sub(r"[ \t]+", " ", l).strip() for l in p.split("\n")]
        lines = [l for l in lines if l and not PAGE_NUM.fullmatch(l)]
        text = LABEL.sub("\n\n", "\n".join(lines))            # blank line = hard boundary
        for block in text.split("\n\n"):
            for s in SPLIT.split(block):
                s = s.strip()
                if not s: continue
                if len(s) > 280 and s.count("\n") >= 2:
                    ls = s.split("\n")
                    chunks = [" ".join(ls[i:i + VERSE_LINES]) for i in range(0, len(ls), VERSE_LINES)]
                else:
                    chunks = [s.replace("\n", " ")]
                for c in chunks:
                    c = re.sub(r"\s+", " ", c).strip()
                    if len(SINHALA.findall(c)) >= 4:
                        bp = pi + 1 - offset
                        units.append({"sid": len(units), "pdf_page": pi + 1,
                                      "book_page": bp if bp >= 1 else "", "text": c})
    return units

for b in BOOKS:
    if "pages" not in b: continue
    b["units"] = split_units(b["pages"], b["page_offset"])
    with open(f'{b["out"]}/{b["book_id"]}_sentences.jsonl', "w", encoding="utf-8") as f:
        for u in b["units"]:
            f.write(json.dumps({"book_id": b["book_id"], **u}, ensure_ascii=False) + "\n")
    with open(f'{b["out"]}/{b["book_id"]}_corpus.txt', "w", encoding="utf-8") as f:   # one unit per line
        f.write("\n".join(u["text"] for u in b["units"]))
    print(f'{b["book_id"]}: {len(b["units"]):,} sentence units -> {b["book_id"]}_corpus.txt')

sinhala_lit_g10_11: 3,311 sentence units -> sinhala_lit_g10_11_corpus.txt


7. Spot-check sheet

In [15]:
SPOT_N, SEED = 100, 42
for b in BOOKS:
    if "units" not in b: continue
    sample = random.Random(SEED).sample(b["units"], min(SPOT_N, len(b["units"])))
    pd.DataFrame([{"sid": u["sid"], "pdf_page": u["pdf_page"], "book_page": u["book_page"],
                   "text": u["text"], "ok (y/n)": "", "error type": "", "note": ""}
                  for u in sorted(sample, key=lambda u: u["sid"])]
                 ).to_excel(f'{b["out"]}/{b["book_id"]}_spot_check.xlsx', index=False)
    print(f'{b["book_id"]}: {len(sample)} units -> {b["book_id"]}_spot_check.xlsx')

sinhala_lit_g10_11: 100 units -> sinhala_lit_g10_11_spot_check.xlsx


In [16]:
import importlib.metadata as md
for b in BOOKS:
    if "units" not in b: continue
    meta = {k: b[k] for k in ["book_id", "pdf", "title", "publisher", "grade", "year", "url", "page_offset", "encoding"]}
    meta.update({
        "method": "pdftotext text layer; FM-Abhaya converted to Unicode with pandukabhaya (fm_abhaya); NFC"
                  if b["encoding"] == "fm" else "pdftotext text layer (already Unicode); NFC",
        "quote_fix": "FM glyph “ kept as an opening quote before conversion" if b["encoding"] == "fm" else "",
        "pdftotext": subprocess.run(["pdftotext", "-v"], capture_output=True, text=True).stderr.splitlines()[0],
        "pandukabhaya": md.version("pandukabhaya"),
        "pages": len(b["pages"]), "sentence_units": len(b["units"]),
        "date": datetime.date.today().isoformat(),
    })
    json.dump(meta, open(f'{b["out"]}/{b["book_id"]}_meta.json', "w", encoding="utf-8"), ensure_ascii=False, indent=1)
    print(json.dumps(meta, ensure_ascii=False, indent=1))

{
 "book_id": "sinhala_lit_g10_11",
 "pdf": "sinhala lit g-10-11.pdf",
 "title": "සිංහල සාහිත්‍ය සංග්‍රහය (Grade 10–11)",
 "publisher": "Educational Publications Department",
 "grade": "10-11",
 "year": "",
 "url": "",
 "page_offset": 10,
 "encoding": "fm",
 "method": "pdftotext text layer; FM-Abhaya converted to Unicode with pandukabhaya (fm_abhaya); NFC",
 "quote_fix": "FM glyph “ kept as an opening quote before conversion",
 "pdftotext": "pdftotext version 24.02.0",
 "pandukabhaya": "1.0.1",
 "pages": 140,
 "sentence_units": 3311,
 "date": "2026-10-14"
}
